In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

features = pd.read_csv('/kaggle/input/cna-hackathon/training_set_features.csv')
labels = pd.read_csv('/kaggle/input/cna-hackathon/training_set_labels.csv')

print("Features shape:", features.shape)
print("Labels shape:", labels.shape)

## Data Overview

In [ ]:
print("Feature columns:\n", features.columns.tolist())
print("\nLabel columns:\n", labels.columns.tolist())

In [ ]:
train = pd.merge(features, labels, on='respondent_id')
train.to_csv('train.csv', index=False)
print("Merged dataset shape:", train.shape)
train.head()

## Data Preprocessing

In [ ]:
columns_to_drop = ['hhs_geo_region', 'employment_industry', 'employment_occupation']
train = train.drop(columns=columns_to_drop)
print("Remaining columns:", train.shape[1])
train.isnull().sum()

### Preprocessing function (reused for train and test)

In [ ]:
def preprocess_df(df):
    df = df.copy()

    # Age group: convert range strings to midpoint integer
    def convert_age_group(age_group):
        if pd.isna(age_group):
            return np.nan
        if age_group == '65+ Years':
            return 65
        parts = age_group.split(' - ')
        return (int(parts[0]) + int(parts[1].split(' ')[0])) // 2

    df['age_group'] = df['age_group'].apply(convert_age_group)

    # Education: ordinal encoding
    df['education'] = df['education'].fillna('< 12 Years')
    education_mapping = {
        '< 12 Years': 1,
        '12 Years': 2,
        'Some College': 3,
        'College Graduate': 4
    }
    df['education'] = df['education'].map(education_mapping)

    # Drop high-missingness columns
    df.drop(columns=['income_poverty', 'health_insurance'], inplace=True, errors='ignore')

    # Categorical columns: impute with proportional random sampling, then one-hot encode
    for col in ['marital_status', 'rent_or_own']:
        if df[col].isnull().any():
            vc = df[col].value_counts()
            nan_idx = df[df[col].isnull()].index
            df.loc[nan_idx, col] = np.random.choice(
                vc.index, size=len(nan_idx), p=vc.values / vc.sum()
            )

    # Employment status: impute with mode
    if 'employment_status' in df.columns and df['employment_status'].isnull().any():
        df['employment_status'] = df['employment_status'].fillna(df['employment_status'].mode()[0])

    # One-hot encode remaining categoricals
    df = pd.get_dummies(df, columns=['race', 'sex', 'marital_status', 'rent_or_own',
                                     'employment_status', 'census_msa'], dtype=int)

    # Numeric columns: fill missing with random draw from observed values
    columns_to_exclude = ['doctor_recc_xyz', 'doctor_recc_seasonal',
                          'xyz_vaccine', 'seasonal_vaccine']
    for col in df.columns:
        if col not in columns_to_exclude and df[col].isnull().any():
            if pd.api.types.is_numeric_dtype(df[col]):
                observed = df[col].dropna()
                fill_vals = observed.sample(df[col].isnull().sum(), replace=True)
                fill_vals.index = df[df[col].isnull()].index
                df[col] = df[col].fillna(fill_vals)

    return df

train = preprocess_df(train)
print("Preprocessed train shape:", train.shape)
print("Remaining nulls:\n", train.isnull().sum()[train.isnull().sum() > 0])

### Impute doctor recommendation columns using logistic regression

In [ ]:
def impute_doctor_recc(df):
    df = df.copy()
    feature_cols = [c for c in df.columns if c not in
                    ['doctor_recc_xyz', 'doctor_recc_seasonal', 'xyz_vaccine', 'seasonal_vaccine']]

    imputer = SimpleImputer(strategy='most_frequent')
    X_base = imputer.fit_transform(df[feature_cols])

    for target in ['doctor_recc_xyz', 'doctor_recc_seasonal']:
        if target not in df.columns:
            continue
        mask = df[target].isnull()
        if mask.sum() == 0:
            continue
        y = df.loc[~mask, target]
        X_train_imp = imputer.transform(df.loc[~mask, feature_cols])
        clf = Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('clf', LogisticRegression(max_iter=500))
        ])
        clf.fit(X_train_imp, y)
        df.loc[mask, target] = clf.predict(imputer.transform(df.loc[mask, feature_cols]))

    return df

train = impute_doctor_recc(train)
print("Nulls after doctor_recc imputation:", train.isnull().sum().sum())

## Exploratory Analysis

In [ ]:
# Drop respondent_id and move targets to end
if 'respondent_id' in train.columns:
    train.drop(columns=['respondent_id'], inplace=True)

targets = ['xyz_vaccine', 'seasonal_vaccine']
train = train[[c for c in train.columns if c not in targets] + targets]

# Correlation heatmap
correlation_matrix = train.corr()
correlation_targets = correlation_matrix[targets]

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_targets, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, center=0, linewidths=0.5)
plt.title('Feature Correlations with Vaccine Uptake', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Vaccine uptake distribution
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col in zip(axes, targets):
    ax.hist(train[col], bins=30, edgecolor='black', color='steelblue', alpha=0.8)
    ax.set_title(f'{col} probability distribution')
    ax.set_xlabel('Predicted probability')
    ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## Model Training

In [ ]:
X = train.drop(columns=targets)
y_xyz = train['xyz_vaccine']
y_seasonal = train['seasonal_vaccine']

X_train, X_test, y_xyz_train, y_xyz_test, y_seasonal_train, y_seasonal_test = train_test_split(
    X, y_xyz, y_seasonal, test_size=0.2, random_state=42
)

print("Train size:", X_train.shape[0], "| Test size:", X_test.shape[0])

In [ ]:
# Logistic Regression pipeline
pipeline_lr = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Random Forest pipeline
pipeline_rf = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Gradient Boosting pipeline
pipeline_gb = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('classifier', GradientBoostingClassifier(random_state=42))
])

param_grid_lr = {
    'classifier__C': [0.1, 1, 10],
    'classifier__solver': ['lbfgs', 'liblinear']
}

param_grid_rf = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [None, 10, 20]
}

param_grid_gb = {
    'classifier__n_estimators': [100, 200],
    'classifier__learning_rate': [0.05, 0.1],
    'classifier__max_depth': [3, 5]
}

results = {}

for name, pipeline, param_grid in [
    ('Logistic Regression', pipeline_lr, param_grid_lr),
    ('Random Forest', pipeline_rf, param_grid_rf),
    ('Gradient Boosting', pipeline_gb, param_grid_gb)
]:
    print(f"Training {name}...")
    gs_xyz = GridSearchCV(pipeline, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
    gs_seasonal = GridSearchCV(pipeline, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)

    gs_xyz.fit(X_train, y_xyz_train)
    gs_seasonal.fit(X_train, y_seasonal_train)

    auc_xyz = roc_auc_score(y_xyz_test, gs_xyz.best_estimator_.predict_proba(X_test)[:, 1])
    auc_seasonal = roc_auc_score(y_seasonal_test, gs_seasonal.best_estimator_.predict_proba(X_test)[:, 1])

    results[name] = {
        'model_xyz': gs_xyz.best_estimator_,
        'model_seasonal': gs_seasonal.best_estimator_,
        'auc_xyz': auc_xyz,
        'auc_seasonal': auc_seasonal,
        'mean_auc': (auc_xyz + auc_seasonal) / 2
    }
    print(f"  XYZ AUC: {auc_xyz:.4f} | Seasonal AUC: {auc_seasonal:.4f} | Mean: {results[name]['mean_auc']:.4f}")

print("\nDone.")

In [ ]:
# Model comparison chart
model_names = list(results.keys())
xyz_aucs = [results[m]['auc_xyz'] for m in model_names]
seasonal_aucs = [results[m]['auc_seasonal'] for m in model_names]

x = np.arange(len(model_names))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, xyz_aucs, width, label='XYZ Vaccine', color='steelblue')
bars2 = ax.bar(x + width/2, seasonal_aucs, width, label='Seasonal Vaccine', color='coral')

ax.set_ylabel('ROC AUC Score')
ax.set_title('Model Comparison by ROC AUC')
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.set_ylim(0.7, 1.0)
ax.legend()
ax.bar_label(bars1, fmt='%.3f', padding=3)
ax.bar_label(bars2, fmt='%.3f', padding=3)
plt.tight_layout()
plt.show()

In [ ]:
# Select best model
best_name = max(results, key=lambda m: results[m]['mean_auc'])
best_model_xyz = results[best_name]['model_xyz']
best_model_seasonal = results[best_name]['model_seasonal']

print(f"Best model: {best_name}")
print(f"  XYZ Vaccine AUC:      {results[best_name]['auc_xyz']:.4f}")
print(f"  Seasonal Vaccine AUC: {results[best_name]['auc_seasonal']:.4f}")
print(f"  Mean AUC:             {results[best_name]['mean_auc']:.4f}")

## Feature Importance

In [ ]:
def plot_feature_importance(model, feature_names, title, top_n=20):
    clf = model.named_steps['classifier']
    if hasattr(clf, 'feature_importances_'):
        importances = clf.feature_importances_
    elif hasattr(clf, 'coef_'):
        importances = np.abs(clf.coef_[0])
    else:
        print("Feature importance not available for this model.")
        return

    feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(top_n)

    plt.figure(figsize=(10, 6))
    feat_imp.sort_values().plot(kind='barh', color='steelblue')
    plt.title(title, fontsize=13)
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()

plot_feature_importance(best_model_xyz, X.columns, f'Top 20 Features - XYZ Vaccine ({best_name})')
plot_feature_importance(best_model_seasonal, X.columns, f'Top 20 Features - Seasonal Vaccine ({best_name})')

## Generate Predictions on Test Set

In [ ]:
test_raw = pd.read_csv('/kaggle/input/cna-hackathon/test_set_features.csv')
res_id = test_raw['respondent_id'].copy()

columns_to_drop = ['hhs_geo_region', 'employment_industry', 'employment_occupation']
test_raw = test_raw.drop(columns=columns_to_drop)

test = preprocess_df(test_raw)
test = impute_doctor_recc(test)

if 'respondent_id' in test.columns:
    test = test.drop(columns=['respondent_id'])

# Align columns with training set
missing_cols = set(X.columns) - set(test.columns)
for col in missing_cols:
    test[col] = 0
test = test[X.columns]

print("Test shape:", test.shape)
print("Nulls:", test.isnull().sum().sum())

In [ ]:
y_test_prob_xyz = best_model_xyz.predict_proba(test)[:, 1]
y_test_prob_seasonal = best_model_seasonal.predict_proba(test)[:, 1]

submission = pd.DataFrame({
    'respondent_id': res_id,
    'xyz_vaccine': y_test_prob_xyz,
    'seasonal_vaccine': y_test_prob_seasonal
})

print(submission.head(10))
submission.to_csv('submission.csv', index=False)
print("\nsubmission.csv saved.")